In [17]:
from idlelib import debugger_r
from json import encoder

import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [18]:
df = pd.read_csv("Gaming_Academic_Performance.csv")

In [19]:
print(df.head())
print(df.info())

   student_id  age  gender  gaming_hours  study_hours  sleep_hours  \
0           1   22    Male          7.23         8.78         6.96   
1           2   19    Male          0.07         8.72         7.63   
2           3   23  Female          1.73         9.56         4.40   
3           4   20  Female          6.62         1.68         7.83   
4           5   22  Female          5.36         5.83         5.55   

   attendance gaming_genre  social_activity  device_usage  reaction_time_ms  \
0       91.44          FPS             3.25          9.36            235.84   
1       63.63       Casual             1.02          3.21            328.71   
2       83.26       Casual             3.46          5.56            313.61   
3       75.04          RPG             1.46         11.78            241.84   
4       65.57          FPS             1.01          8.23            249.31   

   addiction_score stress_level     grades  
0            14.69          Low  86.459555  
1             

Эсперимент 1 (опускаем категориальные, не применяем никаких изменений)

In [20]:
print(df.columns)

Index(['student_id', 'age', 'gender', 'gaming_hours', 'study_hours',
       'sleep_hours', 'attendance', 'gaming_genre', 'social_activity',
       'device_usage', 'reaction_time_ms', 'addiction_score', 'stress_level',
       'grades'],
      dtype='object')


In [21]:
X = df[["age","gaming_hours","study_hours","sleep_hours","attendance","social_activity","device_usage","reaction_time_ms", "addiction_score"]]
y = df["grades"]

X_train, X_test, y_train ,y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [22]:
mlflow.set_experiment("Grades_prediction")
with mlflow.start_run():
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    MSE = mean_squared_error(y_test, y_pred)
    mlflow.log_metric("mse", MSE)
    mlflow.sklearn.log_model(model, "base model")

2026/06/02 18:50:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 18:50:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [23]:
print(f"{MSE:.2f}")

49.83


Проверяем скейлеры (нормализацию)

In [24]:
from sklearn.preprocessing import StandardScaler

In [25]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [26]:
with mlflow.start_run(run_name="Scaled_model"):
    model2 = LinearRegression()
    model2.fit(X_train_scaled, y_train)
    y_pred2 = model2.predict(X_test_scaled)
    MSE2 = mean_squared_error(y_test, y_pred2)

    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("mse", MSE2)
    mlflow.sklearn.log_model(model2, "model_scaled")

2026/06/02 18:50:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 18:50:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [27]:
print(MSE, MSE2)

49.83120287913146 49.83120287913146


скейлер ничего не меняет, модель находит компенсирующие веса

проверяем закодированные

In [28]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

In [29]:
encoder = OneHotEncoder(sparse_output=False, drop="first")
X_cat = encoder.fit_transform(df[["gender", "gaming_genre","stress_level"]])
X_num = X

X_comb = np.hstack([X_cat, X_num])

In [30]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_comb, y, test_size = 0.2, random_state = 42)

In [31]:
with mlflow.start_run(run_name="Encoded_model"):
    model3 = LinearRegression()
    model3.fit(X_train2, y_train2)
    y_pred3 = model3.predict(X_test2)
    MSE3 = mean_squared_error(y_test2, y_pred3)

    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("mse", MSE3)
    mlflow.sklearn.log_model(model3, "model_OHE")

2026/06/02 18:50:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 18:50:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [32]:
print(MSE,MSE3)

49.83120287913146 48.51832499950771


Сделаем дополнительные признаки

In [33]:
print(df.head())

   student_id  age  gender  gaming_hours  study_hours  sleep_hours  \
0           1   22    Male          7.23         8.78         6.96   
1           2   19    Male          0.07         8.72         7.63   
2           3   23  Female          1.73         9.56         4.40   
3           4   20  Female          6.62         1.68         7.83   
4           5   22  Female          5.36         5.83         5.55   

   attendance gaming_genre  social_activity  device_usage  reaction_time_ms  \
0       91.44          FPS             3.25          9.36            235.84   
1       63.63       Casual             1.02          3.21            328.71   
2       83.26       Casual             3.46          5.56            313.61   
3       75.04          RPG             1.46         11.78            241.84   
4       65.57          FPS             1.01          8.23            249.31   

   addiction_score stress_level     grades  
0            14.69          Low  86.459555  
1             

In [34]:
df["gaming_to_study_ratio"] = df["gaming_hours"] / df["study_hours"]
print(df[["gaming_to_study_ratio"]].head(10))

   gaming_to_study_ratio
0               0.823462
1               0.008028
2               0.180962
3               3.940476
4               0.919383
5               1.046784
6               0.062162
7               0.703976
8               0.573770
9               0.068396


In [35]:
X4 = df[["age","gaming_hours","study_hours","sleep_hours","attendance","social_activity","device_usage","reaction_time_ms", "addiction_score","gaming_to_study_ratio"]]
X_comb2 = np.hstack([X_cat, X4])
y4 = df["grades"]

In [36]:
X_train4, X_test4, y_train4, y_test4 = train_test_split(X_comb2, y4, test_size = 0.2, random_state = 42)
with mlflow.start_run(run_name="featured_model"):
    model4 = LinearRegression()
    model4.fit(X_train4, y_train4)
    y_pred4 = model4.predict(X_test4)
    MSE4 = mean_squared_error(y_test4, y_pred4)
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("mse", MSE4)
    mlflow.sklearn.log_model(model4, "model_featured")

2026/06/02 18:50:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 18:50:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Изменится ли что-то если поделить теперь study_hours на  gaming_hours

In [41]:
df["study_to_gaming_ratio"] = df["study_hours"] / df["gaming_hours"]
# Заменяем inf и -inf на 0
df["study_to_gaming_ratio"] = df["study_to_gaming_ratio"].replace([np.inf, -np.inf], 0)

# Или заполняем NaN, если они появились
df["study_to_gaming_ratio"] = df["study_to_gaming_ratio"].fillna(0)

In [42]:
X5 = df[["age","gaming_hours","study_hours","sleep_hours","attendance","social_activity","device_usage","reaction_time_ms", "addiction_score","study_to_gaming_ratio"]]
X_comb3 = np.hstack([X_cat, X5])
y5 = df["grades"]

In [43]:
X_train5, X_test5, y_train5, y_test5 = train_test_split(X_comb3, y5, test_size = 0.2, random_state = 42)
with mlflow.start_run(run_name="featured_model2"):
    model5 = LinearRegression()
    model5.fit(X_train5, y_train5)
    y_pred5 = model5.predict(X_test5)
    MSE5 = mean_squared_error(y_test5, y_pred5)
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("mse", MSE5)
    mlflow.sklearn.log_model(model5, "model_featured_otherway")

2026/06/02 18:51:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 18:51:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [44]:
print(MSE4, MSE5)

47.88281991986694 48.42038661904182


Тест показал, что важно что на что делить

In [50]:
corr1 = df['gaming_to_study_ratio'].corr(df['grades'])
corr2 = df['study_to_gaming_ratio'].corr(df['grades'])
corr3 = df["gaming_hours"].corr(df['grades'])

print(f"Корреляция gaming_to_study_ratio с grades: {corr1:.3f}")
print(f"Корреляция study_to_gaming_ratio с grades: {corr2:.3f}")
print(corr3)

Корреляция gaming_to_study_ratio с grades: -0.776
Корреляция study_to_gaming_ratio с grades: 0.185
-0.5513117134349624
